# Volcanic Eruptions

**Category:** Data Cleaning  
**Dataset:** volcanic-eruptions.csv (3,724 rows) + volcano-list.csv (1,281 rows)  
**Objective:** Parse dates correctly, merge two linked datasets, and find
currently-erupting volcanoes and the longest recorded eruptions in history.

---

In [2]:
# Goal: Identify currently-erupting volcanoes and the longest eruptions in 
# history, by correctly parsing dates and merging two linked datasets.

import pandas as pd
import matplotlib.pyplot as plt

eruptions = pd.read_csv('../data/volcanic-eruptions.csv')
volcanoes = pd.read_csv('../data/volcano-list.csv')

eruptions.head(3)

,volcano_id,start_date,end_date
0,211020,07-1913,04-1944
1,211020,02-1864,11-1868
2,211020,12-1854,05-1855


In [3]:
volcanoes.head(3)

,volcano_id,volcano_name,country,volcanic_region_group,volcanic_region,volcano_landform,primary_volcano_type,activity_evidence,last_known_eruption,latitude,longitude,elevation_m,tectonic_setting,dominant_rock_type
0,210010,West Eifel Volcanic Field,Germany,European Volcanic Regions,Central European Volcanic Province,Cluster,Volcanic field,Eruption Dated,8300 BCE,50.170,6.850,600,Rift zone / Continental crust (>25 km),Foidite
1,210020,Chaine des Puys,France,European Volcanic Regions,Western European Volcanic Province,Cluster,Lava dome(s),Eruption Dated,4040 BCE,45.786,2.981,1464,Rift zone / Continental crust (>25 km),Basalt / Picro-Basalt
2,210030,Olot Volcanic Field,Spain,European Volcanic Regions,Western European Volcanic Province,Cluster,Volcanic field,Evidence Credible,Unknown,42.170,2.530,893,Intraplate / Continental crust (>25 km),Trachybasalt / Tephrite Basanite


## 1. Dataset Exploration

Exploring both files independently before attempting anything with them
together. Checking whether the volcano_id counts roughly correspond
between the two files is a useful early sanity check.

In [4]:
print("=== Eruptions ===")
print("Shape: ", eruptions.shape)
print("Dtypes: ")
print(eruptions.dtypes)
print()

print("=== Volcanoes ===")
print("Shape: ", volcanoes.shape)
print("Missing Values: ")
print(volcanoes.isnull().sum())
print()

print("Unique volcano_ids in eruptions: ", eruptions['volcano_id'].nunique())
print("Total number of rows in volcano-list: ", len(volcanoes))

=== Eruptions ===
Shape:  (3724, 3)
Dtypes: 
volcano_id    int64
start_date      str
end_date        str
dtype: object

=== Volcanoes ===
Shape:  (1281, 14)
Missing Values: 
volcano_id                0
volcano_name              0
country                   0
volcanic_region_group     0
volcanic_region           0
volcano_landform          0
primary_volcano_type      0
activity_evidence         0
last_known_eruption       0
latitude                  0
longitude                 0
elevation_m               0
tectonic_setting          5
dominant_rock_type       25
dtype: int64

Unique volcano_ids in eruptions:  426
Total number of rows in volcano-list:  1281


## 2. Cleaning: Parse Dates

The format is consistently MM-YYYY 
(e.g. "07-1913"). Passing an explicit format string to pd.to_datetime()
is faster and safer than letting pandas guess, guessing can silently
misparse ambiguous formats or run much slower.

In [10]:
eruptions['start_parsed'] = pd.to_datetime(
    eruptions['start_date'], 
    format='%M-%Y'
)
eruptions['end_parsed'] = pd.to_datetime(
    eruptions['end_date'], 
    format='%M-%Y'
)

eruptions[['start_date', 'start_parsed', 'end_date', 'end_parsed']].head()

,start_date,start_parsed,end_date,end_parsed
0,07-1913,1913-01-01 00:07:00,04-1944,1944-01-01 00:04:00
1,02-1864,1864-01-01 00:02:00,11-1868,1868-01-01 00:11:00
2,12-1854,1854-01-01 00:12:00,05-1855,1855-01-01 00:05:00
3,12-1855,1855-01-01 00:12:00,12-1861,1861-01-01 00:12:00
4,07-1824,1824-01-01 00:07:00,09-1834,1834-01-01 00:09:00


## 3. Feature Engineering: Eruption Duration

Computing how many days each eruption lasted, using the two parsed
date columns.

In [11]:
eruptions['duration_days'] = (
    eruptions['end_parsed'] - eruptions['start_parsed']
).dt.days

eruptions[['volcano_id', 'start_date', 'end_date', 'duration_days']].head()

,volcano_id,start_date,end_date,duration_days
0,211020,07-1913,04-1944,11321
1,211020,02-1864,11-1868,1461
2,211020,12-1854,05-1855,364
3,211020,12-1855,12-1861,2192
4,211020,07-1824,09-1834,3653


## 4. Cleaning: Reduce Columns Before Merging

volcano-list.csv has 14 columns; only a few are needed here. Reducing the column
first keeps the merged result readable.

In [16]:
volcano_info = volcanoes[['volcano_id', 'volcano_name', 'country', 'primary_volcano_type']]
eruption_subset = eruptions[['volcano_id', 'start_date', 'end_date', 'duration_days']]

volcano_info.head()

,volcano_id,volcano_name,country,primary_volcano_type
0,210010,West Eifel Volcanic Field,Germany,Volcanic field
1,210020,Chaine des Puys,France,Lava dome(s)
2,210030,Olot Volcanic Field,Spain,Volcanic field
3,210040,Calatrava Volcanic Field,Spain,Volcanic field
4,211004,Colli Albani,Italy,Caldera


In [13]:
eruption_subset.head()

,volcano_id,start_date,end_date,duration_days
0,211020,07-1913,04-1944,11321
1,211020,02-1864,11-1868,1461
2,211020,12-1854,05-1855,364
3,211020,12-1855,12-1861,2192
4,211020,07-1824,09-1834,3653


## 5. Merge the Two Tables

Using how='left' from eruptions into
volcano_info, so no eruption record is silently dropped even if a
volcano_id has no match. Checking for null matches afterward
rather than assuming the merge went cleanly.

In [18]:
merged = eruption_subset.merge(volcano_info, on='volcano_id', how='left')

print("Merged Shape:", merged.shape)
print("Eruption records with no matching volcano:", merged['volcano_name'].isnull().sum())

Merged Shape: (3724, 7)
Eruption records with no matching volcano: 0


Zero unmatched rows indicates that every eruption record found its volcano. This confirms that the merge was clean.